# Simulating stochastic Petri nets

`zrth.simulate` runs a hybrid SPN module. A run is a sequence of steps: the clocks flow at
their rates until the first invariant stops time (a flow written `ite(clk >= 0, rate, None)`
has no value past `clk == 0`, so the step must end there), then every atom takes a round:
transitions whose clock expired fire, the others stutter. The default scheduler, `last()`,
flows straight to that instant; `first(gap)` adds a round every `gap`.

The net is a birth-death process: a birth transition, a death transition and one place.
It is written twice, by hand in `zrth.sugar` and with `zrth.petri`, and run twice: drifting,
and as a stable queue.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from zrth import SPN, Clock, Event, Nat, Var
from zrth import Module as compose
from zrth.petri import Arc, Place, Transition
from zrth.sugar import Module, d, ite, fired, exp
from zrth.simulate import simulate, first

## The net, by hand

The variables first: the time reference the clocks run against, a clock and an event per
transition, and the place's token count. Then a transition owns a clock, armed with an
exponential delay at its rate, that runs down against `t` and is re-armed when it expires,
and an event that it toggles when it fires; the place is a token count awaiting the two
events, and it alone decides what a firing does.

In [ ]:
t, bclk, dclk = Var(Clock()), Var(Clock()), Var(Clock())
bth, dth = Var(Event()), Var(Event())
n = Var(Nat())


def birth_death(lam, mu):
    class Birth(Module):
        def init(self, t):
            return exp(lam), False

        def next(self, clk, bth, t):
            fires = clk == 0
            return ite(fires, exp(lam), clk), ite(fires, ~bth, bth)

        def flow(self, clk, bth, t):
            return ite(clk >= 0, -1 * d(t), None), 0

    class Death(Module):
        def init(self, t):
            return exp(mu), False

        def next(self, clk, dth, t):
            fires = clk == 0
            return ite(fires, exp(mu), clk), ite(fires, ~dth, dth)

        def flow(self, clk, dth, t):
            return ite(clk >= 0, -1 * d(t), None), 0

    class Place(Module):
        def init(self, bth, dth):
            return 0

        def next(self, n, bth, dth):
            born, died = fired(bth), fired(dth)
            return ite(born & ~died, n + 1, ite(died & ~born & (n != 0), n - 1, n))

    net = compose(
        Birth(theory=SPN, ctrl=(bclk, bth), extl=(t,)),
        Death(theory=SPN, ctrl=(dclk, dth), extl=(t,)),
        Place(theory=SPN, ctrl=(n,), extl=(bth, dth)),
        hide={bclk, dclk},
    )
    return net

## The same net with `zrth.petri`

`Transition` and `Place` are those modules, generated from the rate, the initial marking and
the arcs. An arc carries the event the transition fires and, from a place to a transition,
the Bool the place keeps equal to "there are tokens for it": a production arc goes from the
transition into the place, a consumption arc from the place to the transition that takes the
token. The clocks run against `zrth.petri.TIME` unless told otherwise, the clock is private
(reachable as `birth.clock` to plot it), and the place owns its count.

In [ ]:
def petri_birth_death(lam, mu):
    bth, dth = Arc("production"), Arc("consumption")
    birth = Transition(rate=lam, outputs=[bth])
    death = Transition(rate=mu, inputs=[dth])
    place = Place(initial=0, inputs=[bth], outputs=[dth])
    net = compose(birth, death, place)
    return net, birth, death, place

## Plotting a trace

A trace holds `times`, the state after each round (`states`) and the state each round saw
(`before`). A count holds its value until the next round; a clock is a straight line from
what the last round produced down to what the next round saw, exactly `0.0` at an expiry.

In [ ]:
def step_plot(ax, trace, var, **kw):
    ax.step(trace.times, trace.series(var), where="post", **kw)


def sawtooth(ax, trace, clock, **kw):
    xs, ys = [], []
    for i, time in enumerate(trace.times):
        if trace.before[i] is not None:
            xs.append(time)
            ys.append(trace.before[i][clock])
        xs.append(time)
        ys.append(trace.states[i][clock])
    ax.plot(xs, ys, **kw)

## Births at 2, deaths at 1

Births twice as frequent as deaths: the count drifts upwards at about one token per unit of
time.

In [ ]:
system = birth_death(lam=2.0, mu=1.0)
trace = simulate(system, t=t, horizon=20.0, seed=1)
print(f"{len(trace) - 1} rounds up to t = {trace.times[-1]}")

petri_net, _, _, petri_place = petri_birth_death(lam=2.0, mu=1.0)
petri_trace = simulate(petri_net, horizon=20.0, seed=1)
print("the two spellings give the same run:", petri_trace.times == trace.times and petri_trace.series(petri_place.count) == trace.series(n))

52 rounds up to t = 20.0
the two spellings give the same run: True


In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
step_plot(top, trace, n, color="black")
top.set_ylabel("tokens")
sawtooth(bottom, trace, bclk, label="birth clock")
sawtooth(bottom, trace, dclk, label="death clock")
bottom.set_ylabel("time to expiry")
bottom.set_xlabel("t")
bottom.legend()
plt.show()

## Births at 1, deaths at 2: a stable queue

With the rates swapped the net is an M/M/1 queue with load ρ = λ/μ = 0.5, whose token count
has a known stationary law: the fraction of time spent with `n` tokens is (1 − ρ) ρⁿ. A long
run shows it.

In [ ]:
system, birth, death, place = petri_birth_death(lam=1.0, mu=2.0)
trace = simulate(system, horizon=1000.0, seed=2)
print(f"{len(trace) - 1} rounds up to t = {trace.times[-1]}")

3064 rounds up to t = 1000.0


In [ ]:
def time_in_each_state(trace, var):
    held = Counter()
    for i in range(len(trace) - 1):
        held[trace.states[i][var]] += trace.times[i + 1] - trace.times[i]
    total = sum(held.values())
    return {k: v / total for k, v in sorted(held.items())}


rho = 1.0 / 2.0
empirical = time_in_each_state(trace, place.count)
ks = range(0, max(empirical) + 1)

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
step_plot(left, trace, place.count, color="black", linewidth=0.8)
left.set_xlim(0, 100)
left.set_xlabel("t")
left.set_ylabel("tokens (first 100 time units)")
right.bar(ks, [empirical.get(k, 0.0) for k in ks], label="fraction of time (simulated)")
right.plot(ks, [(1 - rho) * rho**k for k in ks], "o-", color="black", label="(1 − ρ) ρⁿ")
right.set_xlabel("n")
right.legend()
plt.show()

## Seeing the steps

With `first(gap)` there is a round every `gap` as well as at every expiry; most of them
stutter, and the trace shows the clock counting down.

In [ ]:
system, birth, death, place = petri_birth_death(lam=2.0, mu=1.0)
trace = simulate(system, horizon=4.0, seed=1, scheduler=first(0.25))
print(f"{len(trace) - 1} rounds up to t = {trace.times[-1]}")

22 rounds up to t = 4.0


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sawtooth(ax, trace, birth.clock, marker=".", label="birth clock")
step_plot(ax, trace, place.count, color="black", label="tokens")
ax.set_xlabel("t")
ax.legend()
plt.show()